In [8]:
# Cell 1 — Delta monitor session (no Spark required)
from pathlib import Path

import pandas as pd
from deltalake import DeltaTable

BASE_DIR = Path.cwd().resolve()
PROJECT_ROOT = BASE_DIR.parent if BASE_DIR.name.lower() == 'spark' else BASE_DIR
BRONZE_PATH = PROJECT_ROOT / 'delta' / 'bronze' / 'btc_trades'
SILVER_PATH = PROJECT_ROOT / 'delta' / 'silver' / 'btc_aggregates'

def load_delta_table(path: Path) -> pd.DataFrame:
    table = DeltaTable(str(path))
    return table.to_pandas()

print('Monitor session ready.')
print(f'Bronze path: {BRONZE_PATH}')
print(f'Silver path: {SILVER_PATH}')

Monitor session ready.
Bronze path: C:\Users\franc\Desktop\DEV\Real-Time Crypto Pipeline\delta\bronze\btc_trades
Silver path: C:\Users\franc\Desktop\DEV\Real-Time Crypto Pipeline\delta\silver\btc_aggregates


In [9]:
# Cell 2 — Latest bronze rows (re-run to refresh)
try:
    bronze_df = load_delta_table(BRONZE_PATH)
    if 'event_time' in bronze_df.columns:
        bronze_df['event_time'] = pd.to_datetime(bronze_df['event_time'], errors='coerce')
        bronze_df = bronze_df.sort_values('event_time', ascending=False)
    display(bronze_df.head(20))
except Exception as e:
    print(f'Bronze data not available yet: {e}')

,symbol,price,volume,event_time
140,BTCUSDT,66984.74,0.00034,2026-04-05 15:22:19.047000+00:00
87,BTCUSDT,66984.74,0.00006,2026-04-05 15:22:19.047000+00:00
84,BTCUSDT,66984.74,0.00007,2026-04-05 15:22:19.047000+00:00
85,BTCUSDT,66984.74,0.00016,2026-04-05 15:22:19.047000+00:00
86,BTCUSDT,66984.74,0.00016,2026-04-05 15:22:19.047000+00:00
45,BTCUSDT,66984.74,0.00001,2026-04-05 15:22:19.040000+00:00
44,BTCUSDT,66984.74,0.00008,2026-04-05 15:22:19.040000+00:00
43,BTCUSDT,66984.74,0.00008,2026-04-05 15:22:19.040000+00:00
82,BTCUSDT,66984.74,0.00008,2026-04-05 15:22:19.040000+00:00
83,BTCUSDT,66984.74,0.00008,2026-04-05 15:22:19.040000+00:00


In [7]:
# Cell 3 — Latest silver aggregations (re-run to refresh)
try:
    silver_df = load_delta_table(SILVER_PATH)
    if 'window_start' in silver_df.columns:
        silver_df['window_start'] = pd.to_datetime(silver_df['window_start'], errors='coerce')
        silver_df = silver_df.sort_values('window_start', ascending=False)
    display(silver_df.head(10))
except Exception as e:
    print(f'Silver data not available yet: {e}')

,window_start,window_end,symbol,avg_price,total_volume
0,2026-04-04 22:57:00+00:00,2026-04-04 22:58:00+00:00,BTCUSDT,67388.815573,0.64864
1,2026-04-04 22:56:00+00:00,2026-04-04 22:57:00+00:00,BTCUSDT,67387.998480,0.44647


In [23]:
# Cell 4 — Delta table history (audit log)
try:
    bronze_table = DeltaTable(str(BRONZE_PATH))
    history_df = pd.DataFrame(bronze_table.history())
    if not history_df.empty and 'timestamp' in history_df.columns:
        history_df = history_df.sort_values('timestamp', ascending=False)
    display(history_df[['timestamp', 'operation', 'version']].head(20) if not history_df.empty else history_df)
except Exception as e:
    print(f'Bronze history not available yet: {e}')

,timestamp,operation,version
0,1775343470671,STREAMING UPDATE,10
1,1775343460775,STREAMING UPDATE,9
2,1775343450752,STREAMING UPDATE,8
3,1775343440790,STREAMING UPDATE,7
4,1775343430846,STREAMING UPDATE,6
5,1775343420777,STREAMING UPDATE,5
6,1775343410779,STREAMING UPDATE,4
7,1775343403167,STREAMING UPDATE,3
8,1775343401108,STREAMING UPDATE,2
9,1775343390383,STREAMING UPDATE,1


In [24]:
# Cell 5 — Row count per symbol
try:
    bronze_df = load_delta_table(BRONZE_PATH)
    if 'symbol' in bronze_df.columns:
        display(
            bronze_df.groupby('symbol', dropna=False)
            .size()
            .reset_index(name='count')
            .sort_values('count', ascending=False)
        )
    else:
        print('Column "symbol" not found in bronze table.')
except Exception as e:
    print(f'Bronze data not available yet: {e}')

,symbol,count
0,BTCUSDT,1003
